In [1]:
import json
from collections import Counter, defaultdict, deque
from pathlib import Path

import numpy as np
import pandas as pd

np.random.seed(42)
pd.set_option('display.max_columns', None)


## Task 1: Build a Complete ETL Pipeline


In [2]:
# Deterministic synthetic generation is used so validation and comparisons remain reproducible.
products = [
    'Laptop', 'Mouse', 'Keyboard', 'Monitor', 'Headphones',
    'Webcam', 'USB Cable', 'Desk Lamp', 'Phone Case', 'Charger'
]
shipping_countries = ['usa', 'canada', 'mexico', 'germany', 'france', 'japan']

raw_records = []
for i in range(200):
    order_date = pd.Timestamp('2025-01-01') + pd.to_timedelta(np.random.randint(0, 120), unit='D')
    raw_records.append({
        'order_id': f'ORD{1000 + i}',
        'customer_id': f'CUST{100 + np.random.randint(0, 30)}',
        'product_name': np.random.choice(products),
        'quantity': int(np.random.randint(1, 6)),
        'unit_price': float(np.round(np.random.uniform(5, 120), 2)),
        'order_date': order_date.strftime('%Y-%m-%d'),
        'shipping_country': np.random.choice(shipping_countries)
    })

# Inject required problematic records by category.
raw_records[5]['product_name'] = None
raw_records[37]['product_name'] = ''
raw_records[89]['product_name'] = None

raw_records[12]['quantity'] = -2
raw_records[58]['unit_price'] = -10.0
raw_records[144]['quantity'] = -1

raw_records[22]['order_date'] = 'not-a-date'
raw_records[75]['order_date'] = '2025-13-45'
raw_records[163]['order_date'] = '31/31/2025'

raw_records[31]['order_id'] = raw_records[0]['order_id']
raw_records[132]['order_id'] = raw_records[1]['order_id']
raw_records[177]['order_id'] = raw_records[2]['order_id']

raw_records[45]['quantity'] = '3'
raw_records[111]['unit_price'] = '19.99'
raw_records[190]['quantity'] = '2'

raw_data_preview = pd.DataFrame(raw_records).head(8)
raw_data_preview


,order_id,customer_id,product_name,quantity,unit_price,order_date,shipping_country
0,ORD1000,CUST119,Desk Lamp,5,73.64,2025-04-13,canada
1,ORD1001,CUST122,Desk Lamp,5,74.13,2025-03-24,mexico
2,ORD1002,CUST120,Mouse,4,112.93,2025-01-22,canada
3,ORD1003,CUST127,Headphones,1,39.99,2025-03-05,japan
4,ORD1004,CUST124,Laptop,3,75.36,2025-04-18,canada
5,ORD1005,CUST127,NaN,4,64.14,2025-04-02,usa
6,ORD1006,CUST104,Keyboard,5,56.81,2025-01-03,canada
7,ORD1007,CUST124,Mouse,2,40.03,2025-01-04,france


In [3]:
def extract(raw_records):
    valid_records = []
    rejected_records = []

    required_fields = [
        'order_id', 'customer_id', 'product_name',
        'quantity', 'unit_price', 'order_date', 'shipping_country'
    ]

    for record in raw_records:
        reason = None

        if not isinstance(record, dict):
            reason = 'record_not_dict'
        elif any(field not in record for field in required_fields):
            reason = 'missing_required_field'
        elif record['product_name'] is None or str(record['product_name']).strip() == '':
            reason = 'missing_product_name'
        else:
            quantity = record['quantity']
            unit_price = record['unit_price']

            is_quantity_number = isinstance(quantity, (int, float, np.integer, np.floating)) and not isinstance(quantity, bool)
            is_price_number = isinstance(unit_price, (int, float, np.integer, np.floating)) and not isinstance(unit_price, bool)

            if not (is_quantity_number and is_price_number):
                reason = 'non_numeric_quantity_or_unit_price'
            elif quantity <= 0 or unit_price <= 0:
                reason = 'non_positive_quantity_or_unit_price'
            else:
                try:
                    parsed_date = pd.to_datetime(record['order_date'], errors='raise')
                except Exception:
                    reason = 'malformed_order_date'

        if reason is None:
            cleaned_record = record.copy()
            cleaned_record['order_date'] = pd.to_datetime(cleaned_record['order_date']).strftime('%Y-%m-%d')
            valid_records.append(cleaned_record)
        else:
            rejected_records.append({'record': record, 'reason': reason})

    reject_counts = pd.Series([r['reason'] for r in rejected_records]).value_counts().rename_axis('reason').reset_index(name='count')
    stage_summary = pd.DataFrame({
        'metric': ['raw_records', 'valid_records', 'rejected_records'],
        'count': [len(raw_records), len(valid_records), len(rejected_records)]
    })

    return valid_records, rejected_records, reject_counts, stage_summary


valid_records, rejected_records, extract_reject_counts, extract_stage_summary = extract(raw_records)

{
    'extract_stage_summary': extract_stage_summary,
    'extract_rejection_counts': extract_reject_counts,
    'sample_rejected_records': pd.DataFrame(rejected_records).head(5)
}


{'extract_stage_summary':              metric  count
 0       raw_records    200
 1     valid_records    188
 2  rejected_records     12,
 'extract_rejection_counts':                                 reason  count
 0                 missing_product_name      3
 1  non_positive_quantity_or_unit_price      3
 2                 malformed_order_date      3
 3   non_numeric_quantity_or_unit_price      3,
 'sample_rejected_records':                                               record  \
 0  {'order_id': 'ORD1005', 'customer_id': 'CUST12...   
 1  {'order_id': 'ORD1012', 'customer_id': 'CUST11...   
 2  {'order_id': 'ORD1022', 'customer_id': 'CUST12...   
 3  {'order_id': 'ORD1037', 'customer_id': 'CUST11...   
 4  {'order_id': 'ORD1045', 'customer_id': 'CUST11...   
 
                                 reason  
 0                 missing_product_name  
 1  non_positive_quantity_or_unit_price  
 2                 malformed_order_date  
 3                 missing_product_name  
 4   non_numeric_

In [4]:
def transform(valid_records):
    df = pd.DataFrame(valid_records).copy()

    df['order_date'] = pd.to_datetime(df['order_date'])
    df['quantity'] = df['quantity'].astype(int)
    df['unit_price'] = df['unit_price'].astype(float)

    df['total_amount'] = df['quantity'] * df['unit_price']
    df['order_month'] = df['order_date'].dt.to_period('M').astype(str)
    df['order_day_of_week'] = df['order_date'].dt.day_name()
    df['shipping_country'] = df['shipping_country'].astype(str).str.strip().str.title()

    before_dedup = len(df)
    # Dedupe in transform preserves raw ingestion while enforcing analytical uniqueness.
    df = df.drop_duplicates(subset=['order_id'], keep='first').reset_index(drop=True)
    duplicates_removed = before_dedup - len(df)

    df['amount_category'] = np.select(
        [df['total_amount'] < 25, df['total_amount'].between(25, 100, inclusive='both')],
        ['small', 'medium'],
        default='large'
    )

    transform_summary = pd.DataFrame({
        'metric': ['input_valid_records', 'duplicates_removed', 'transformed_records'],
        'count': [before_dedup, duplicates_removed, len(df)]
    })

    return df, transform_summary


clean_df, transform_summary = transform(valid_records)

{
    'transform_summary': transform_summary,
    'clean_data_preview': clean_df.head(8)
}


{'transform_summary':                 metric  count
 0  input_valid_records    188
 1   duplicates_removed      3
 2  transformed_records    185,
 'clean_data_preview':   order_id customer_id product_name  quantity  unit_price order_date  \
 0  ORD1000     CUST119    Desk Lamp         5       73.64 2025-04-13   
 1  ORD1001     CUST122    Desk Lamp         5       74.13 2025-03-24   
 2  ORD1002     CUST120        Mouse         4      112.93 2025-01-22   
 3  ORD1003     CUST127   Headphones         1       39.99 2025-03-05   
 4  ORD1004     CUST124       Laptop         3       75.36 2025-04-18   
 5  ORD1006     CUST104     Keyboard         5       56.81 2025-01-03   
 6  ORD1007     CUST124        Mouse         2       40.03 2025-01-04   
 7  ORD1008     CUST119    USB Cable         4       61.95 2025-01-02   
 
   shipping_country  total_amount order_month order_day_of_week amount_category  
 0           Canada        368.20     2025-04            Sunday           large  
 1       

In [5]:
def load(df, path):
    path = Path(path)
    df.to_parquet(path, index=False)
    loaded_df = pd.read_parquet(path)

    verification = pd.DataFrame({
        'check': ['row_count_match', 'dtypes_match'],
        'result': [
            len(df) == len(loaded_df),
            df.dtypes.astype(str).equals(loaded_df.dtypes.astype(str))
        ]
    })

    return loaded_df, verification


loaded_df, load_verification = load(clean_df, 'orders_clean.parquet')

pipeline_summary = pd.DataFrame({
    'stage': ['raw', 'valid', 'transformed', 'loaded'],
    'record_count': [len(raw_records), len(valid_records), len(clean_df), len(loaded_df)]
})

{
    'pipeline_summary': pipeline_summary,
    'load_verification': load_verification
}


{'pipeline_summary':          stage  record_count
 0          raw           200
 1        valid           188
 2  transformed           185
 3       loaded           185,
 'load_verification':              check  result
 0  row_count_match    True
 1     dtypes_match    True}

## Task 2: ETL vs ELT Comparison


In [9]:
def extract_elt(raw_records):
    accepted_records = []
    rejected_records = []

    for record in raw_records:
        try:
            if not isinstance(record, dict):
                raise ValueError('record_not_dict')
            json.dumps(record)
            accepted_records.append(record)
        except Exception:
            rejected_records.append({'record': record, 'reason': 'unparseable_record'})

    return accepted_records, rejected_records


def load_elt(raw_records, path):
    path = Path(path)
    # Store each record as JSON text so raw mixed types survive parquet typing constraints.
    lake_df = pd.DataFrame({'raw_record_json': [json.dumps(record) for record in raw_records]})
    lake_df.to_parquet(path, index=False)
    return path


def transform_elt(path):
    lake_df = pd.read_parquet(path)
    lake_records = [json.loads(item) for item in lake_df['raw_record_json'].tolist()]
    valid_records_elt, rejected_records_elt, reject_counts_elt, _ = extract(lake_records)
    transformed_elt_df, transform_summary_elt = transform(valid_records_elt)

    return transformed_elt_df, rejected_records_elt, reject_counts_elt, transform_summary_elt


elt_extracted_records, elt_rejected_extract = extract_elt(raw_records)
elt_path = load_elt(elt_extracted_records, 'orders_lake_raw.parquet')
elt_df, elt_rejected_transform, elt_reject_counts_transform, elt_transform_summary = transform_elt(elt_path)

etl_elt_comparison = pd.DataFrame({
    'approach': ['ETL', 'ELT'],
    'records_after_extract': [len(valid_records), len(elt_extracted_records)],
    'records_in_destination': [len(loaded_df), len(elt_df)],
    'rejected_early': [len(rejected_records), len(elt_rejected_extract)],
    'rejected_during_transform': [0, len(elt_rejected_transform)]
})

{
    'etl_elt_comparison': etl_elt_comparison,
    'elt_rejection_counts_transform': elt_reject_counts_transform
}



{'etl_elt_comparison':   approach  records_after_extract  records_in_destination  rejected_early  \
 0      ETL                    188                     185              12   
 1      ELT                    200                     185               0   
 
    rejected_during_transform  
 0                          0  
 1                         12  ,
 'elt_rejection_counts_transform':                                 reason  count
 0                 missing_product_name      3
 1  non_positive_quantity_or_unit_price      3
 2                 malformed_order_date      3
 3   non_numeric_quantity_or_unit_price      3}

### ETL vs ELT Analysis

- ETL caught data quality issues during extraction, so only validated records moved into analytical storage.
- ELT loaded nearly everything first, then rejected bad records at transform time after reading from the data lake.
- ETL advantage: stronger downstream quality guarantees and less propagation of bad data.
- ELT advantage: faster ingestion and ability to reprocess raw history with updated rules.
- Choose ETL when strict quality gates are needed before storage consumers read data.
- Choose ELT when ingestion speed, schema flexibility, and reprocessing are priorities.


## Task 3: Simulate Modes of Dataflow


In [10]:
def compute_customer_features(orders):
    if not orders:
        return {}

    df = pd.DataFrame(orders).copy()
    if 'total_amount' not in df.columns:
        df['total_amount'] = df['quantity'] * df['unit_price']

    df['order_date'] = pd.to_datetime(df['order_date'])

    grouped = df.groupby('customer_id', as_index=False).agg(
        total_orders=('order_id', 'count'),
        average_amount=('total_amount', 'mean'),
        last_order_date=('order_date', 'max')
    )

    grouped['last_order_date'] = grouped['last_order_date'].dt.strftime('%Y-%m-%d')
    grouped['average_amount'] = grouped['average_amount'].round(2)

    return {
        row['customer_id']: {
            'total_orders': int(row['total_orders']),
            'average_amount': float(row['average_amount']),
            'last_order_date': row['last_order_date']
        }
        for _, row in grouped.iterrows()
    }


orders_for_dataflow = loaded_df.head(20).to_dict(orient='records')

# Through a database
database = {'orders': [], 'features': {}}

def order_service_write(order):
    database['orders'].append(order)

def feature_service_compute():
    database['features'] = compute_customer_features(database['orders'])

def prediction_service_read(customer_id):
    return database['features'].get(customer_id)

for order in orders_for_dataflow:
    order_service_write(order)
feature_service_compute()

db_features = database['features']

# Step 2: Through services
service_order_store = []

def order_service_write_api(order):
    service_order_store.append(order)

def order_service_get_orders():
    return list(service_order_store)

def feature_service_get_features():
    orders = order_service_get_orders()
    return compute_customer_features(orders)

def prediction_service_request(customer_id):
    return feature_service_get_features().get(customer_id)

for order in orders_for_dataflow:
    order_service_write_api(order)

service_features = feature_service_get_features()

# Step 3: Through a message broker
class SimpleBroker:
    def __init__(self):
        self.topics = defaultdict(list)
        self.subscribers = defaultdict(list)

    def publish(self, topic, message):
        self.topics[topic].append(message)
        for callback in self.subscribers[topic]:
            callback(message)

    def subscribe(self, topic, callback):
        self.subscribers[topic].append(callback)

    def get_latest(self, topic):
        return self.topics[topic][-1] if self.topics[topic] else None


broker = SimpleBroker()
orders_state = []
feature_state = {}
prediction_cache = {}

def on_order(message):
    orders_state.append(message)
    features = compute_customer_features(orders_state)
    broker.publish('features', features)

def on_feature(message):
    prediction_cache.clear()
    prediction_cache.update(message)

broker.subscribe('orders', on_order)
broker.subscribe('features', on_feature)

for order in orders_for_dataflow:
    broker.publish('orders', order)

broker_features = broker.get_latest('features')

all_customer_ids = sorted(set([o['customer_id'] for o in orders_for_dataflow]))
feature_match = all(
    db_features.get(cid) == service_features.get(cid) == broker_features.get(cid)
    for cid in all_customer_ids
)

comparison_rows = []
for cid in all_customer_ids[:8]:
    comparison_rows.append({
        'customer_id': cid,
        'database_mode': db_features.get(cid),
        'services_mode': service_features.get(cid),
        'broker_mode': broker_features.get(cid)
    })

{
    'feature_match_across_modes': feature_match,
    'mode_feature_sample': pd.DataFrame(comparison_rows)
}


{'feature_match_across_modes': True,
 'mode_feature_sample':   customer_id                                      database_mode  \
 0     CUST100  {'total_orders': 1, 'average_amount': 583.75, ...   
 1     CUST104  {'total_orders': 3, 'average_amount': 264.22, ...   
 2     CUST106  {'total_orders': 1, 'average_amount': 27.85, '...   
 3     CUST109  {'total_orders': 1, 'average_amount': 36.64, '...   
 4     CUST111  {'total_orders': 2, 'average_amount': 114.06, ...   
 5     CUST112  {'total_orders': 1, 'average_amount': 362.4, '...   
 6     CUST116  {'total_orders': 2, 'average_amount': 225.0, '...   
 7     CUST119  {'total_orders': 2, 'average_amount': 308.0, '...   
 
                                        services_mode  \
 0  {'total_orders': 1, 'average_amount': 583.75, ...   
 1  {'total_orders': 3, 'average_amount': 264.22, ...   
 2  {'total_orders': 1, 'average_amount': 27.85, '...   
 3  {'total_orders': 1, 'average_amount': 36.64, '...   
 4  {'total_orders': 2, 'average

### Dataflow Mode Comparison

- Database mode is tightly coupled to shared storage schema; latency is moderate and state is centralized.
- Service-to-service mode is contract-driven and decouples persistence, but availability of upstream services directly impacts downstream requests.
- Broker mode is event-driven and loosely coupled; it supports near-real-time updates and buffering, but consumers may lag when they are down.


## Task 4: Batch Processing vs Stream Processing


In [11]:
def batch_daily_aggregates(df):
    batch_df = df.copy()
    batch_df['order_date'] = pd.to_datetime(batch_df['order_date'])
    batch_df['order_day'] = batch_df['order_date'].dt.date

    daily_base = batch_df.groupby('order_day', as_index=False).agg(
        total_orders=('order_id', 'count'),
        total_revenue=('total_amount', 'sum'),
        average_order_size=('total_amount', 'mean'),
        unique_customers=('customer_id', 'nunique')
    )

    product_revenue = (
        batch_df.groupby(['order_day', 'product_name'], as_index=False)['total_amount']
        .sum()
        .rename(columns={'total_amount': 'product_revenue'})
    )

    top_product = (
        product_revenue.sort_values(['order_day', 'product_revenue', 'product_name'], ascending=[True, False, True])
        .drop_duplicates(subset=['order_day'], keep='first')
        .loc[:, ['order_day', 'product_name']]
        .rename(columns={'product_name': 'top_product'})
    )

    result = daily_base.merge(top_product, on='order_day', how='left').sort_values('order_day').reset_index(drop=True)
    result['total_revenue'] = result['total_revenue'].round(2)
    result['average_order_size'] = result['average_order_size'].round(2)

    return result


batch_results = batch_daily_aggregates(loaded_df)
batch_results.head(10)


,order_day,total_orders,total_revenue,average_order_size,unique_customers,top_product
0,2025-01-01,4,735.25,183.81,4,USB Cable
1,2025-01-02,2,268.51,134.26,2,USB Cable
2,2025-01-03,3,555.23,185.08,3,Keyboard
3,2025-01-04,1,80.06,80.06,1,Mouse
4,2025-01-05,4,186.69,46.67,4,Charger
5,2025-01-06,3,463.45,154.48,3,USB Cable
6,2025-01-08,2,518.87,259.44,2,Monitor
7,2025-01-09,1,18.78,18.78,1,Headphones
8,2025-01-12,1,80.10,80.10,1,Headphones
9,2025-01-13,1,79.30,79.30,1,Mouse


In [12]:
class StreamProcessor:
    def __init__(self, window_size=50):
        self.window_size = window_size
        self.total_orders = 0
        self.total_revenue = 0.0
        self.unique_customers = set()
        self.window_amounts = deque(maxlen=window_size)
        self.window_products = deque(maxlen=window_size)
        self.window_product_counter = Counter()

    def process_order(self, order):
        amount = float(order.get('total_amount', order['quantity'] * order['unit_price']))
        product = order['product_name']

        if len(self.window_products) == self.window_size:
            oldest_product = self.window_products[0]
            self.window_product_counter[oldest_product] -= 1
            if self.window_product_counter[oldest_product] == 0:
                del self.window_product_counter[oldest_product]

        self.total_orders += 1
        self.total_revenue += amount
        self.unique_customers.add(order['customer_id'])

        self.window_amounts.append(amount)
        self.window_products.append(product)
        self.window_product_counter[product] += 1

    def current_stats(self):
        window_avg = float(np.mean(self.window_amounts)) if self.window_amounts else 0.0
        top_product = None
        if self.window_product_counter:
            top_product = sorted(self.window_product_counter.items(), key=lambda x: (-x[1], x[0]))[0][0]

        return {
            'running_total_orders': self.total_orders,
            'running_total_revenue': round(self.total_revenue, 2),
            'window_avg_order_size_last_50': round(window_avg, 2),
            'running_unique_customers': len(self.unique_customers),
            'current_most_popular_product_last_50': top_product
        }


stream_processor = StreamProcessor(window_size=50)
stream_checkpoints = []

stream_orders = loaded_df.to_dict(orient='records')
for idx, order in enumerate(stream_orders, start=1):
    stream_processor.process_order(order)
    if idx % 50 == 0:
        checkpoint = stream_processor.current_stats().copy()
        checkpoint['processed_records'] = idx
        stream_checkpoints.append(checkpoint)

stream_checkpoints_df = pd.DataFrame(stream_checkpoints)
final_stream_stats = stream_processor.current_stats()

final_batch_totals = {
    'batch_total_orders': int(batch_results['total_orders'].sum()),
    'batch_total_revenue': round(float(batch_results['total_revenue'].sum()), 2)
}

stream_batch_match = {
    'total_orders_match': final_batch_totals['batch_total_orders'] == final_stream_stats['running_total_orders'],
    'total_revenue_match': np.isclose(final_batch_totals['batch_total_revenue'], final_stream_stats['running_total_revenue'])
}

{
    'stream_checkpoints': stream_checkpoints_df,
    'final_batch_totals': final_batch_totals,
    'final_stream_stats': final_stream_stats,
    'cumulative_match': stream_batch_match
}


{'stream_checkpoints':    running_total_orders  running_total_revenue  window_avg_order_size_last_50  \
 0                    50               10590.86                         211.82   
 1                   100               20883.56                         205.85   
 2                   150               29987.10                         182.07   
 
    running_unique_customers current_most_popular_product_last_50  \
 0                        23                             Keyboard   
 1                        28                           Headphones   
 2                        29                           Headphones   
 
    processed_records  
 0                 50  
 1                100  
 2                150  ,
 'final_batch_totals': {'batch_total_orders': 185,
  'batch_total_revenue': 36322.89},
 'final_stream_stats': {'running_total_orders': 185,
  'running_total_revenue': 36322.89,
  'window_avg_order_size_last_50': 192.36,
  'running_unique_customers': 30,
  'current_most_pop

### Batch vs Stream Analysis

- Cumulative totals align because both methods process the same validated records.
- Windowed stream statistics differ from daily batch statistics because stream windows are order-count based (last 50) rather than calendar-time buckets.
- Batch output gives stable period summaries for reporting and finance.
- Stream output gives low-latency behavior signals useful for monitoring and real-time decisions.


## Task 5: Combine Batch and Stream Features


In [13]:
product_category_map = {
    'Laptop': 'Electronics',
    'Mouse': 'Accessories',
    'Keyboard': 'Accessories',
    'Monitor': 'Electronics',
    'Headphones': 'Accessories',
    'Webcam': 'Accessories',
    'USB Cable': 'Accessories',
    'Desk Lamp': 'Home',
    'Phone Case': 'Mobile',
    'Charger': 'Mobile'
}

feature_df = loaded_df.copy()
feature_df['order_date'] = pd.to_datetime(feature_df['order_date'])
feature_df['category'] = feature_df['product_name'].map(product_category_map).fillna('Other')

reference_time = feature_df['order_date'].max()
sample_customers = (
    feature_df['customer_id']
    .value_counts()
    .head(5)
    .index
    .tolist()
)


def compute_batch_features(customer_df, reference_time):
    most_purchased_category = (
        customer_df['category'].value_counts().sort_values(ascending=False).index[0]
        if not customer_df.empty else None
    )
    return {
        'total_lifetime_orders': int(len(customer_df)),
        'avg_order_amount': round(float(customer_df['total_amount'].mean()), 2) if not customer_df.empty else 0.0,
        'days_since_first_order': int((reference_time - customer_df['order_date'].min()).days) if not customer_df.empty else None,
        'most_purchased_category': most_purchased_category
    }


def compute_stream_features(customer_df, reference_time):
    recent_df = customer_df.sort_values('order_date').tail(10)
    recent_top_category = (
        recent_df['category'].value_counts().sort_values(ascending=False).index[0]
        if not recent_df.empty else None
    )
    seconds_since_last_order = (
        int((reference_time - recent_df['order_date'].max()).total_seconds())
        if not recent_df.empty else None
    )
    return {
        'recent_order_count': int(len(recent_df)),
        'recent_avg_amount': round(float(recent_df['total_amount'].mean()), 2) if not recent_df.empty else 0.0,
        'seconds_since_last_order': seconds_since_last_order,
        'recent_top_category': recent_top_category
    }


combined_feature_vectors = {}
for customer_id in sample_customers:
    customer_df = feature_df[feature_df['customer_id'] == customer_id].copy()
    batch_features = compute_batch_features(customer_df, reference_time)
    stream_features = compute_stream_features(customer_df, reference_time)
    combined_feature_vectors[customer_id] = {
        **batch_features,
        **stream_features
    }

combined_features_df = pd.DataFrame.from_dict(combined_feature_vectors, orient='index').reset_index().rename(columns={'index': 'customer_id'})
combined_features_df


,customer_id,total_lifetime_orders,avg_order_amount,days_since_first_order,most_purchased_category,recent_order_count,recent_avg_amount,seconds_since_last_order,recent_top_category
0,CUST104,11,268.11,119,Accessories,10,275.39,2073600,Accessories
1,CUST112,11,197.85,117,Accessories,10,202.49,345600,Accessories
2,CUST100,11,236.05,102,Accessories,10,246.60,1296000,Accessories
3,CUST129,11,118.46,115,Accessories,10,128.27,1209600,Accessories
4,CUST125,11,130.20,118,Accessories,10,141.15,259200,Accessories


### Why Combine Batch and Stream Features

- Batch features capture long-term customer behavior and baseline preferences.
- Streaming features capture immediate intent shifts and recency effects.
- Example where batch alone is insufficient: real-time fraud spikes need second-level recency signals.
- Example where stream alone is insufficient: credit or lifetime value scoring needs long-horizon purchase history.


## Short Summary

A full notebook was implemented with ETL and ELT pipelines, three dataflow modes, batch vs stream processing, and combined batch-stream customer feature vectors. The required function names, variable names, file names, and validation/feature rules were followed, with outputs shown via notebook cells instead of `print`.
